# Decoding Deception — Colab training

Thin shell around `src/models/train.py`. Edit the `EXPERIMENT` and `REPO_URL` below.

All training logic lives in the repo so this notebook stays a one-shot.

In [ ]:
# 1. Mount Google Drive (optional, for persistent checkpoints / data)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Sync repo. Defensive: validates Drive path before linking, cleans stale
# symlinks from prior runs, prints what's in Drive if the path is wrong.
import os, subprocess
REPO_URL    = ''                                  # leave '' to use the Drive copy
DRIVE_REPO  = '/content/drive/MyDrive/CSS2'       # adjust to your Drive folder name
WORK        = '/content/CSS2'

if REPO_URL:
    if not os.path.exists(WORK):
        subprocess.check_call(['git', 'clone', REPO_URL, WORK])
else:
    assert os.path.isdir(DRIVE_REPO), (
        f'DRIVE_REPO not found: {DRIVE_REPO}\n'
        f'Top of Drive: {os.listdir("/content/drive/MyDrive/")[:20]}'
    )
    if os.path.lexists(WORK):
        if os.path.islink(WORK):
            os.unlink(WORK)                       # stale symlink from prior run
        else:
            raise SystemExit(f'{WORK} exists and is not a symlink; remove it manually')
    os.symlink(DRIVE_REPO, WORK)

os.chdir(WORK)
print('cwd =', os.getcwd())
print('contents:', os.listdir('.')[:12])

In [ ]:
# 3. Install deps
!pip install -q -r requirements.txt

In [ ]:
# 4. Set caches to Drive so re-runs don't redownload weights
%env HF_HOME=/content/drive/MyDrive/hf_cache
%env TRANSFORMERS_CACHE=/content/drive/MyDrive/hf_cache
%env TORCH_HOME=/content/drive/MyDrive/torch_cache

In [ ]:
# 5. (Optional) Build data pipeline. Skip if data/processed/ is already on Drive.
!python -m src.data.load_existing --all
!python -m src.data.map_labels
!python -m src.data.preprocess
!python -m src.data.split
!python -m src.data.balance

In [ ]:
# 6. Train
EXPERIMENT = 'experiments/text_xlmr_existing.yaml'
!python -m src.models.train --config {EXPERIMENT}

In [ ]:
# 7. Evaluate latest run
from pathlib import Path
runs = sorted(Path('results/runs').glob('*'))
latest = str(runs[-1])
print('latest run:', latest)
!python -m src.evaluation.metrics --run-dir {latest}
!python -m src.evaluation.significance bootstrap --run {latest} --B 1000

In [ ]:
# 8. Persist results back to Drive
import shutil, time
dst = f'/content/drive/MyDrive/propaganda-lt-results-{int(time.time())}.zip'
shutil.make_archive(dst.replace('.zip', ''), 'zip', 'results')
print('wrote', dst)